# Baseline ResNet18 — Results
Loads from `checkpoints/baseline_best.pt`. No training happens here.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
from config import BASELINE_CKPT, DEVICE
from models import build_resnet18
from train import load_model_weights, load_results
from plot import plot_history
from data import build_loaders
from train import evaluate
import torch.nn as nn
print(f'Device: {DEVICE}')

## Load checkpoint

In [ ]:
ckpt = torch.load(BASELINE_CKPT, map_location=DEVICE)
print(f"Best val acc: {ckpt['best_val_acc']:.4f}")
print(f"Trained for {ckpt['epoch']+1} epochs")

## Training curves

In [ ]:
plot_history(
    ckpt['history'],
    title='ResNet18 Baseline — Training and Validation Curves',
    save_name='baseline_curves.png',
)

## Verify final val accuracy

In [ ]:
model = build_resnet18(pretrained=False)
model = load_model_weights(model, BASELINE_CKPT)
_, val_loader = build_loaders('standard')
criterion = nn.CrossEntropyLoss()
val_loss, val_acc = evaluate(model, val_loader, criterion)
print(f'Val loss: {val_loss:.4f}  |  Val acc: {val_acc:.4f}')

## Error analysis — misclassified examples

In [ ]:
import matplotlib.pyplot as plt
from torchvision import datasets
from config import DATA_ROOT
from data import get_val_transform

val_ds = datasets.CIFAR100(root=DATA_ROOT, train=True, download=True,
                            transform=get_val_transform())
fine_labels = val_ds.classes

model.eval()
shown = 0
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes = axes.flatten()

for x, y in val_loader:
    x, y = x.to(DEVICE), y.to(DEVICE)
    with torch.no_grad():
        preds = model(x).argmax(1)
    wrong = torch.where(preds != y)[0]
    for i in wrong:
        if shown >= 6:
            break
        img = x[i].cpu().permute(1, 2, 0)
        # De-normalize for display
        mean = torch.tensor([0.5071, 0.4867, 0.4408])
        std  = torch.tensor([0.2675, 0.2565, 0.2761])
        img  = (img * std + mean).clamp(0, 1)
        axes[shown].imshow(img)
        axes[shown].set_title(
            f'True: {fine_labels[y[i].item()]}\nPred: {fine_labels[preds[i].item()]}',
            fontsize=8
        )
        axes[shown].axis('off')
        shown += 1
    if shown >= 6:
        break

plt.suptitle('Misclassified Examples')
plt.tight_layout()
plt.savefig('../figures/baseline_errors.png', dpi=150, bbox_inches='tight')
plt.show()